## **EVALUACIÓN ZERO-SHOT**

En este fichero se evalúa el rendimiento de todos los modelos de SAM estudiados en el proyecto utilizando el conjunto de datos ISIC-2016.

En este primer bloque de código se importan las librerías necesarias para la ejecución del fichero completo. Además, se importan las funciones de métricas y de medición de inferencia. Para ello se tuvo que añadir la raíz del proyecto al path de Python para que estos imports funcionen correctamente. También, se define la ruta al datset con el que entrenaremos los modelos junto a sus correspondientes subdivisiones.

Finalmente, se define la función get_bbox_from_mask, que calcula la caja delimitadora a partir de los píxeles activos de una máscara detectando sus contornos, devolviendo None si la máscara está vacía.

In [2]:
from ultralytics import SAM
from ultralytics.models.sam import SAM3SemanticPredictor
import numpy as np
import os
import cv2
import pandas as pd
import time
import sys
import os


root_path = os.path.abspath('..')
if root_path not in sys.path:
    sys.path.append(root_path)

from utils.segmentation_quality_metrics import boundary_iou, resize_for_hausdorff, hausdorff_95, compute_all_metrics
from utils.efficiency_metrics import measure_inference_central_point, measure_inference_sam3_prompt_zero_shot

DATASET_ROOT = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\ISIC_2016"

SPLITS = [
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_GroundTruth"),
    },
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_GroundTruth"),
    },
]

def get_bbox_from_mask(mask_binary):
    contours, _ = cv2.findContours(mask_binary.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    x, y, w, h = cv2.boundingRect(np.vstack(contours))
    return x, y, x + w, y + h


La función **evaluate_model** es idéntica para los modelos: SAM 1 Base, SAM 1 Large, SAM 2 Base, SAM 2 Large, SAM 2.1 Base, SAM 2.1 Large y SAM 3. El dataset se recorre por splits, procesando conjuntamente las imágenes de entrenamiento y test originales. Para cada imagen se obtienela caja delimitadora a partir de los píxeles activos de la máscara mediante get_bbox_from_mask, descartando las imágenes cuya máscara esté vacía, y se calcula el punto central a partir de él. La máscara predicha se redimensiona al tamaño original y se calculan todas las métricas. Los resultados se guardan en un CSV acumulando los de cada modelo.

En el caso de SAM 3, la variante con prompt textual usa la descripción "skin lesion" (lesión en la piel) como prompt, adecuada para el tipo de imágenes de ISIC 2016. Cuando el modelo devuelve varias máscaras, se selecciona directamente la primera, que corresponde a la de mayor confianza según el modelo.

**SAM 1 BASE**

In [3]:
def evaluate_model():
    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam_b.pt")
    model.to("cuda")

    iou_scores         = []
    precision_scores   = []
    recall_scores      = []
    f1_scores          = []
    dice_scores        = []
    specificity_scores = []
    f2_scores          = []
    pixel_accuracy_scores = []
    boundary_iou_scores   = []
    hausdorff_95_scores   = []
    latency_scores     = []
    vram_scores        = []

    for split in SPLITS:
        images_dir = split["images"]
        masks_dir  = split["masks"]

        for mask_filename in sorted(os.listdir(masks_dir)):
            if not mask_filename.lower().endswith(".png"):
                continue

            img_stem   = mask_filename.replace("_Segmentation.png", "")
            img_path   = os.path.join(images_dir, img_stem + ".jpg")
            mask_path  = os.path.join(masks_dir, mask_filename)

            if not os.path.exists(img_path):
                continue

            gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            gt_mask = np.squeeze(gt_mask)
            gt_mask = (gt_mask > 127).astype(bool)

            bbox = get_bbox_from_mask(gt_mask)
            if bbox is None:
                continue

            xmin, ymin, xmax, ymax = bbox
            central_point = [[(xmin + xmax) / 2, (ymin + ymax) / 2]]

            results, latency, vram = measure_inference_central_point(model, img_path, central_point)

            if results[0].masks is None or len(results[0].masks.data) == 0:
                continue

            scores   = results[0].boxes.conf.cpu().numpy()
            best_idx = np.argmax(scores)
            pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

            H, W      = gt_mask.shape
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

            iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
            iou_scores.append(iou)
            precision_scores.append(precision)
            recall_scores.append(recall)
            f1_scores.append(f1)
            dice_scores.append(dice)
            specificity_scores.append(specificity)
            f2_scores.append(f2)
            pixel_accuracy_scores.append(pixel_accuracy)
            boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
            pred_mask, gt_mask = resize_for_hausdorff(pred_mask, gt_mask)
            hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
            latency_scores.append(latency)
            vram_scores.append(vram)

    results = {
        "modelo":             ["sam_b"],
        "mean_iou":           [np.mean(iou_scores)],
        "mean_f1":            [np.mean(f1_scores)],
        "mean_recall":        [np.mean(recall_scores)],
        "mean_precision":     [np.mean(precision_scores)],
        "mean_dice":          [np.mean(dice_scores)],
        "mean_specificity":   [np.mean(specificity_scores)],
        "mean_f2":            [np.mean(f2_scores)],
        "mean_pixel_accuracy":[np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou":  [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95":  [np.mean(hausdorff_95_scores)],
        "mean_latency_ms":    [np.mean(latency_scores)],
        "mean_vram_mb":       [np.mean(vram_scores)],
    }

    return results

start_time = time.time()
results = evaluate_model()
eval_time = time.time() - start_time

results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_isic2016.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)

**SAM 1 LARGE**

In [4]:
def evaluate_model():
    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam_l.pt")
    model.to("cuda")

    iou_scores         = []
    precision_scores   = []
    recall_scores      = []
    f1_scores          = []
    dice_scores        = []
    specificity_scores = []
    f2_scores          = []
    pixel_accuracy_scores = []
    boundary_iou_scores   = []
    hausdorff_95_scores   = []
    latency_scores     = []
    vram_scores        = []

    for split in SPLITS:
        images_dir = split["images"]
        masks_dir  = split["masks"]

        for mask_filename in sorted(os.listdir(masks_dir)):
            if not mask_filename.lower().endswith(".png"):
                continue

            img_stem   = mask_filename.replace("_Segmentation.png", "")
            img_path   = os.path.join(images_dir, img_stem + ".jpg")
            mask_path  = os.path.join(masks_dir, mask_filename)

            if not os.path.exists(img_path):
                continue

            gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            gt_mask = np.squeeze(gt_mask)
            gt_mask = (gt_mask > 127).astype(bool)

            bbox = get_bbox_from_mask(gt_mask)
            if bbox is None:
                continue

            xmin, ymin, xmax, ymax = bbox
            central_point = [[(xmin + xmax) / 2, (ymin + ymax) / 2]]

            results, latency, vram = measure_inference_central_point(model, img_path, central_point)

            if results[0].masks is None or len(results[0].masks.data) == 0:
                continue

            scores   = results[0].boxes.conf.cpu().numpy()
            best_idx = np.argmax(scores)
            pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

            H, W      = gt_mask.shape
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

            iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
            iou_scores.append(iou)
            precision_scores.append(precision)
            recall_scores.append(recall)
            f1_scores.append(f1)
            dice_scores.append(dice)
            specificity_scores.append(specificity)
            f2_scores.append(f2)
            pixel_accuracy_scores.append(pixel_accuracy)
            boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
            pred_mask, gt_mask = resize_for_hausdorff(pred_mask, gt_mask)
            hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
            latency_scores.append(latency)
            vram_scores.append(vram)

    results = {
        "modelo":             ["sam_l"],
        "mean_iou":           [np.mean(iou_scores)],
        "mean_f1":            [np.mean(f1_scores)],
        "mean_recall":        [np.mean(recall_scores)],
        "mean_precision":     [np.mean(precision_scores)],
        "mean_dice":          [np.mean(dice_scores)],
        "mean_specificity":   [np.mean(specificity_scores)],
        "mean_f2":            [np.mean(f2_scores)],
        "mean_pixel_accuracy":[np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou":  [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95":  [np.mean(hausdorff_95_scores)],
        "mean_latency_ms":    [np.mean(latency_scores)],
        "mean_vram_mb":       [np.mean(vram_scores)],
    }

    return results

start_time = time.time()
results = evaluate_model()
eval_time = time.time() - start_time

results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_isic2016.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)

**SAM 2 BASE**

In [5]:
def evaluate_model():
    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam2_b.pt")
    model.to("cuda")

    iou_scores         = []
    precision_scores   = []
    recall_scores      = []
    f1_scores          = []
    dice_scores        = []
    specificity_scores = []
    f2_scores          = []
    pixel_accuracy_scores = []
    boundary_iou_scores   = []
    hausdorff_95_scores   = []
    latency_scores     = []
    vram_scores        = []

    for split in SPLITS:
        images_dir = split["images"]
        masks_dir  = split["masks"]

        for mask_filename in sorted(os.listdir(masks_dir)):
            if not mask_filename.lower().endswith(".png"):
                continue

            img_stem   = mask_filename.replace("_Segmentation.png", "")
            img_path   = os.path.join(images_dir, img_stem + ".jpg")
            mask_path  = os.path.join(masks_dir, mask_filename)

            if not os.path.exists(img_path):
                continue

            gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            gt_mask = np.squeeze(gt_mask)
            gt_mask = (gt_mask > 127).astype(bool)

            bbox = get_bbox_from_mask(gt_mask)
            if bbox is None:
                continue

            xmin, ymin, xmax, ymax = bbox
            central_point = [[(xmin + xmax) / 2, (ymin + ymax) / 2]]

            results, latency, vram = measure_inference_central_point(model, img_path, central_point)

            if results[0].masks is None or len(results[0].masks.data) == 0:
                continue

            scores   = results[0].boxes.conf.cpu().numpy()
            best_idx = np.argmax(scores)
            pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

            H, W      = gt_mask.shape
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

            iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
            iou_scores.append(iou)
            precision_scores.append(precision)
            recall_scores.append(recall)
            f1_scores.append(f1)
            dice_scores.append(dice)
            specificity_scores.append(specificity)
            f2_scores.append(f2)
            pixel_accuracy_scores.append(pixel_accuracy)
            boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
            pred_mask, gt_mask = resize_for_hausdorff(pred_mask, gt_mask)
            hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
            latency_scores.append(latency)
            vram_scores.append(vram)

    results = {
        "modelo":             ["sam2_b"],
        "mean_iou":           [np.mean(iou_scores)],
        "mean_f1":            [np.mean(f1_scores)],
        "mean_recall":        [np.mean(recall_scores)],
        "mean_precision":     [np.mean(precision_scores)],
        "mean_dice":          [np.mean(dice_scores)],
        "mean_specificity":   [np.mean(specificity_scores)],
        "mean_f2":            [np.mean(f2_scores)],
        "mean_pixel_accuracy":[np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou":  [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95":  [np.mean(hausdorff_95_scores)],
        "mean_latency_ms":    [np.mean(latency_scores)],
        "mean_vram_mb":       [np.mean(vram_scores)],
    }

    return results

start_time = time.time()
results = evaluate_model()
eval_time = time.time() - start_time

results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_isic2016.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)

**SAM 2 LARGE**

In [6]:
def evaluate_model():
    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam2_l.pt")
    model.to("cuda")

    iou_scores         = []
    precision_scores   = []
    recall_scores      = []
    f1_scores          = []
    dice_scores        = []
    specificity_scores = []
    f2_scores          = []
    pixel_accuracy_scores = []
    boundary_iou_scores   = []
    hausdorff_95_scores   = []
    latency_scores     = []
    vram_scores        = []

    for split in SPLITS:
        images_dir = split["images"]
        masks_dir  = split["masks"]

        for mask_filename in sorted(os.listdir(masks_dir)):
            if not mask_filename.lower().endswith(".png"):
                continue

            img_stem   = mask_filename.replace("_Segmentation.png", "")
            img_path   = os.path.join(images_dir, img_stem + ".jpg")
            mask_path  = os.path.join(masks_dir, mask_filename)

            if not os.path.exists(img_path):
                continue

            gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            gt_mask = np.squeeze(gt_mask)
            gt_mask = (gt_mask > 127).astype(bool)

            bbox = get_bbox_from_mask(gt_mask)
            if bbox is None:
                continue

            xmin, ymin, xmax, ymax = bbox
            central_point = [[(xmin + xmax) / 2, (ymin + ymax) / 2]]

            results, latency, vram = measure_inference_central_point(model, img_path, central_point)

            if results[0].masks is None or len(results[0].masks.data) == 0:
                continue

            scores   = results[0].boxes.conf.cpu().numpy()
            best_idx = np.argmax(scores)
            pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

            H, W      = gt_mask.shape
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

            iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
            iou_scores.append(iou)
            precision_scores.append(precision)
            recall_scores.append(recall)
            f1_scores.append(f1)
            dice_scores.append(dice)
            specificity_scores.append(specificity)
            f2_scores.append(f2)
            pixel_accuracy_scores.append(pixel_accuracy)
            boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
            pred_mask, gt_mask = resize_for_hausdorff(pred_mask, gt_mask)
            hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
            latency_scores.append(latency)
            vram_scores.append(vram)

    results = {
        "modelo":             ["sam2_l"],
        "mean_iou":           [np.mean(iou_scores)],
        "mean_f1":            [np.mean(f1_scores)],
        "mean_recall":        [np.mean(recall_scores)],
        "mean_precision":     [np.mean(precision_scores)],
        "mean_dice":          [np.mean(dice_scores)],
        "mean_specificity":   [np.mean(specificity_scores)],
        "mean_f2":            [np.mean(f2_scores)],
        "mean_pixel_accuracy":[np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou":  [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95":  [np.mean(hausdorff_95_scores)],
        "mean_latency_ms":    [np.mean(latency_scores)],
        "mean_vram_mb":       [np.mean(vram_scores)],
    }

    return results

start_time = time.time()
results = evaluate_model()
eval_time = time.time() - start_time

results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_isic2016.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)

**SAM 2.1 BASE**

In [7]:
def evaluate_model():
    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam2.1_b.pt")
    model.to("cuda")

    iou_scores         = []
    precision_scores   = []
    recall_scores      = []
    f1_scores          = []
    dice_scores        = []
    specificity_scores = []
    f2_scores          = []
    pixel_accuracy_scores = []
    boundary_iou_scores   = []
    hausdorff_95_scores   = []
    latency_scores     = []
    vram_scores        = []

    for split in SPLITS:
        images_dir = split["images"]
        masks_dir  = split["masks"]

        for mask_filename in sorted(os.listdir(masks_dir)):
            if not mask_filename.lower().endswith(".png"):
                continue

            img_stem   = mask_filename.replace("_Segmentation.png", "")
            img_path   = os.path.join(images_dir, img_stem + ".jpg")
            mask_path  = os.path.join(masks_dir, mask_filename)

            if not os.path.exists(img_path):
                continue

            gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            gt_mask = np.squeeze(gt_mask)
            gt_mask = (gt_mask > 127).astype(bool)

            bbox = get_bbox_from_mask(gt_mask)
            if bbox is None:
                continue

            xmin, ymin, xmax, ymax = bbox
            central_point = [[(xmin + xmax) / 2, (ymin + ymax) / 2]]

            results, latency, vram = measure_inference_central_point(model, img_path, central_point)

            if results[0].masks is None or len(results[0].masks.data) == 0:
                continue

            scores   = results[0].boxes.conf.cpu().numpy()
            best_idx = np.argmax(scores)
            pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

            H, W      = gt_mask.shape
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

            iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
            iou_scores.append(iou)
            precision_scores.append(precision)
            recall_scores.append(recall)
            f1_scores.append(f1)
            dice_scores.append(dice)
            specificity_scores.append(specificity)
            f2_scores.append(f2)
            pixel_accuracy_scores.append(pixel_accuracy)
            boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
            pred_mask, gt_mask = resize_for_hausdorff(pred_mask, gt_mask)
            hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
            latency_scores.append(latency)
            vram_scores.append(vram)

    results = {
        "modelo":             ["sam2.1_b"],
        "mean_iou":           [np.mean(iou_scores)],
        "mean_f1":            [np.mean(f1_scores)],
        "mean_recall":        [np.mean(recall_scores)],
        "mean_precision":     [np.mean(precision_scores)],
        "mean_dice":          [np.mean(dice_scores)],
        "mean_specificity":   [np.mean(specificity_scores)],
        "mean_f2":            [np.mean(f2_scores)],
        "mean_pixel_accuracy":[np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou":  [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95":  [np.mean(hausdorff_95_scores)],
        "mean_latency_ms":    [np.mean(latency_scores)],
        "mean_vram_mb":       [np.mean(vram_scores)],
    }

    return results

start_time = time.time()
results = evaluate_model()
eval_time = time.time() - start_time

results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_isic2016.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)

**SAM 2.1 LARGE**

In [8]:
def evaluate_model():
    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam2.1_l.pt")
    model.to("cuda")

    iou_scores         = []
    precision_scores   = []
    recall_scores      = []
    f1_scores          = []
    dice_scores        = []
    specificity_scores = []
    f2_scores          = []
    pixel_accuracy_scores = []
    boundary_iou_scores   = []
    hausdorff_95_scores   = []
    latency_scores     = []
    vram_scores        = []

    for split in SPLITS:
        images_dir = split["images"]
        masks_dir  = split["masks"]

        for mask_filename in sorted(os.listdir(masks_dir)):
            if not mask_filename.lower().endswith(".png"):
                continue

            img_stem   = mask_filename.replace("_Segmentation.png", "")
            img_path   = os.path.join(images_dir, img_stem + ".jpg")
            mask_path  = os.path.join(masks_dir, mask_filename)

            if not os.path.exists(img_path):
                continue

            gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            gt_mask = np.squeeze(gt_mask)
            gt_mask = (gt_mask > 127).astype(bool)

            bbox = get_bbox_from_mask(gt_mask)
            if bbox is None:
                continue

            xmin, ymin, xmax, ymax = bbox
            central_point = [[(xmin + xmax) / 2, (ymin + ymax) / 2]]

            results, latency, vram = measure_inference_central_point(model, img_path, central_point)

            if results[0].masks is None or len(results[0].masks.data) == 0:
                continue

            scores   = results[0].boxes.conf.cpu().numpy()
            best_idx = np.argmax(scores)
            pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

            H, W      = gt_mask.shape
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

            iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
            iou_scores.append(iou)
            precision_scores.append(precision)
            recall_scores.append(recall)
            f1_scores.append(f1)
            dice_scores.append(dice)
            specificity_scores.append(specificity)
            f2_scores.append(f2)
            pixel_accuracy_scores.append(pixel_accuracy)
            boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
            pred_mask, gt_mask = resize_for_hausdorff(pred_mask, gt_mask)
            hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
            latency_scores.append(latency)
            vram_scores.append(vram)

    results = {
        "modelo":             ["sam2.1_l"],
        "mean_iou":           [np.mean(iou_scores)],
        "mean_f1":            [np.mean(f1_scores)],
        "mean_recall":        [np.mean(recall_scores)],
        "mean_precision":     [np.mean(precision_scores)],
        "mean_dice":          [np.mean(dice_scores)],
        "mean_specificity":   [np.mean(specificity_scores)],
        "mean_f2":            [np.mean(f2_scores)],
        "mean_pixel_accuracy":[np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou":  [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95":  [np.mean(hausdorff_95_scores)],
        "mean_latency_ms":    [np.mean(latency_scores)],
        "mean_vram_mb":       [np.mean(vram_scores)],
    }

    return results

start_time = time.time()
results = evaluate_model()
eval_time = time.time() - start_time

results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_isic2016.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)

**SAM 3**

In [9]:
def evaluate_model():
    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam3.pt")
    model.to("cuda")

    iou_scores         = []
    precision_scores   = []
    recall_scores      = []
    f1_scores          = []
    dice_scores        = []
    specificity_scores = []
    f2_scores          = []
    pixel_accuracy_scores = []
    boundary_iou_scores   = []
    hausdorff_95_scores   = []
    latency_scores     = []
    vram_scores        = []

    for split in SPLITS:
        images_dir = split["images"]
        masks_dir  = split["masks"]

        for mask_filename in sorted(os.listdir(masks_dir)):
            if not mask_filename.lower().endswith(".png"):
                continue

            img_stem   = mask_filename.replace("_Segmentation.png", "")
            img_path   = os.path.join(images_dir, img_stem + ".jpg")
            mask_path  = os.path.join(masks_dir, mask_filename)

            if not os.path.exists(img_path):
                continue

            gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            gt_mask = np.squeeze(gt_mask)
            gt_mask = (gt_mask > 127).astype(bool)

            bbox = get_bbox_from_mask(gt_mask)
            if bbox is None:
                continue

            xmin, ymin, xmax, ymax = bbox
            central_point = [[(xmin + xmax) / 2, (ymin + ymax) / 2]]

            results, latency, vram = measure_inference_central_point(model, img_path, central_point)

            if results[0].masks is None or len(results[0].masks.data) == 0:
                continue

            scores   = results[0].boxes.conf.cpu().numpy()
            best_idx = np.argmax(scores)
            pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

            H, W      = gt_mask.shape
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

            iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
            iou_scores.append(iou)
            precision_scores.append(precision)
            recall_scores.append(recall)
            f1_scores.append(f1)
            dice_scores.append(dice)
            specificity_scores.append(specificity)
            f2_scores.append(f2)
            pixel_accuracy_scores.append(pixel_accuracy)
            boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
            pred_mask, gt_mask = resize_for_hausdorff(pred_mask, gt_mask)
            hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
            latency_scores.append(latency)
            vram_scores.append(vram)

    results = {
        "modelo":             ["sam3"],
        "mean_iou":           [np.mean(iou_scores)],
        "mean_f1":            [np.mean(f1_scores)],
        "mean_recall":        [np.mean(recall_scores)],
        "mean_precision":     [np.mean(precision_scores)],
        "mean_dice":          [np.mean(dice_scores)],
        "mean_specificity":   [np.mean(specificity_scores)],
        "mean_f2":            [np.mean(f2_scores)],
        "mean_pixel_accuracy":[np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou":  [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95":  [np.mean(hausdorff_95_scores)],
        "mean_latency_ms":    [np.mean(latency_scores)],
        "mean_vram_mb":       [np.mean(vram_scores)],
    }

    return results

start_time = time.time()
results = evaluate_model()
eval_time = time.time() - start_time

results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_isic2016.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)

c:\Users\DanielTalavera\AppData\Local\Programs\Python\Python310\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post1)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
c:\Users\DanielTalavera\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must

**SAM 3 USANDO PROMPTS DE TEXTO PARA SEGMENTAR**

In [10]:
TEXT_PROMPT = "skin lesion"

def evaluate_model():
    overrides = dict(
        conf=0.25,
        task="segment",
        mode="predict",
        model="C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam3.pt",
        device="cuda",
    )
    predictor = SAM3SemanticPredictor(overrides=overrides)

    iou_scores            = []
    precision_scores      = []
    recall_scores         = []
    f1_scores             = []
    dice_scores           = []
    specificity_scores    = []
    f2_scores             = []
    pixel_accuracy_scores = []
    boundary_iou_scores   = []
    hausdorff_95_scores   = []
    latency_scores        = []
    vram_scores           = []

    for split in SPLITS:
        images_dir = split["images"]
        masks_dir  = split["masks"]

        for mask_filename in sorted(os.listdir(masks_dir)):
            if not mask_filename.lower().endswith(".png"):
                continue

            img_stem  = mask_filename.replace("_Segmentation.png", "")
            img_path  = os.path.join(images_dir, img_stem + ".jpg")
            mask_path = os.path.join(masks_dir, mask_filename)

            if not os.path.exists(img_path):
                continue

            gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            gt_mask = np.squeeze(gt_mask)
            gt_mask = (gt_mask > 127).astype(bool)
            if gt_mask.sum() == 0:
                continue

            results, latency, vram = measure_inference_sam3_prompt_zero_shot(predictor, img_path, TEXT_PROMPT)

            if results[0].masks is None or len(results[0].masks.data) == 0:
                continue

            masks = results[0].masks.data.cpu().numpy()
            H, W  = gt_mask.shape

            best_idx = 0

            pred_mask = masks[best_idx]
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

            iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
            iou_scores.append(iou)
            precision_scores.append(precision)
            recall_scores.append(recall)
            f1_scores.append(f1)
            dice_scores.append(dice)
            specificity_scores.append(specificity)
            f2_scores.append(f2)
            pixel_accuracy_scores.append(pixel_accuracy)
            boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
            pred_mask, gt_mask = resize_for_hausdorff(pred_mask, gt_mask)
            hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
            latency_scores.append(latency)
            vram_scores.append(vram)

    results = {
        "modelo":              ["sam3_prompt"],
        "mean_iou":            [np.mean(iou_scores)],
        "mean_f1":             [np.mean(f1_scores)],
        "mean_recall":         [np.mean(recall_scores)],
        "mean_precision":      [np.mean(precision_scores)],
        "mean_dice":           [np.mean(dice_scores)],
        "mean_specificity":    [np.mean(specificity_scores)],
        "mean_f2":             [np.mean(f2_scores)],
        "mean_pixel_accuracy": [np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou":   [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95":   [np.mean(hausdorff_95_scores)],
        "mean_latency_ms":     [np.mean(latency_scores)],
        "mean_vram_mb":        [np.mean(vram_scores)],
    }

    return results

start_time = time.time()
results = evaluate_model()
eval_time = time.time() - start_time

results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_isic2016.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)


Ultralytics 8.4.26  Python-3.10.11 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3090, 24575MiB)
WARNING imgsz=[640] must be multiple of max stride 14, updating to [644]

image 1/1 C:\Users\DanielTalavera\Desktop\Trabajo_Fin_de_Grado\ISIC_2016\ISBI2016_ISIC_Part1_Training_Data\ISIC_0000000.jpg: 644x644 3 skin lesions, 249.2ms
Speed: 3.1ms preprocess, 249.2ms inference, 14.5ms postprocess per image at shape (1, 3, 644, 644)
Results saved to C:\Users\DanielTalavera\Desktop\Trabajo_Fin_de_Grado\SAM-zero-shot\runs\segment\predict33
WARNING imgsz=[640] must be multiple of max stride 14, updating to [644]

image 1/1 C:\Users\DanielTalavera\Desktop\Trabajo_Fin_de_Grado\ISIC_2016\ISBI2016_ISIC_Part1_Training_Data\ISIC_0000001.jpg: 644x644 1 skin lesion, 55.9ms
Speed: 3.0ms preprocess, 55.9ms inference, 1.8ms postprocess per image at shape (1, 3, 644, 644)
Results saved to C:\Users\DanielTalavera\Desktop\Trabajo_Fin_de_Grado\SAM-zero-shot\runs\segment\predict33
WARNING imgsz=[640] must be multip